# Lamplighter — notebook workflow

Build the model **and run training** in the browser; the notebook does setup and
gets the results. The loop is:

1. **Start** a session here → 2. **build** the model on the canvas →
3. **register data** with `sess.data(X=X, y=y)` and pick it in the **Data** tab →
4. hit **▶ Run** in the **Training** tab → 5. pull `sess.model` / `sess.history`
back here.

Training executes *in this kernel* — your data never leaves it (the session
holds references, not copies) — running exactly the code the app's preview
panes show.

> This notebook lives in `examples/`; the first cell puts the repo root on
> `sys.path` so `import lamplighter` resolves.

## 1. Start a session

The first run builds the frontend (`npm install` + `npm run build`, ~10s), then
serves the app and opens it in your browser. Subsequent runs are instant.

In [ ]:
import sys
from pathlib import Path

# The repo root (this notebook runs with examples/ as its cwd).
sys.path.insert(0, str(Path.cwd().parent.resolve()))

import lamplighter

sess = lamplighter.start()
sess.url

## 2. Build a model on the canvas

In the browser, build a small **MNIST classifier** (784-pixel inputs → 10 digits):

1. **Input** → **Linear** (Out Features `128`) → **ReLU** → **Linear** (Out
   Features `10`) → **Output**, wired in a chain.
2. Don't worry about the Input's *Shape* — picking your data in the next step
   fills it in automatically.

The shape badges update live as you wire pins.

## 3. Load data here, register it with the session

Run the next cell: it loads MNIST as tensors and hands the session *references*
to them — `sess.data(X=X, y=y)`. Nothing is copied; in-place changes are visible
immediately, and if you recreate a tensor, just re-run the cell to repoint the
name. (`sess.list_data()` shows what's registered; `sess.drop_data("X")` removes.)

Then, on the **System** canvas, add a **Data** node (the sidebar's **DATA** →
**＋ set**) and wire it into your model's input. Select the node to configure it:

1. Source **memory** (the default) lists exactly what you registered — hit
   **↻ refresh** after registering.
2. Under **Input(s)** pick `X`, under **Target(s)** pick `y`. Picking `X` also
   pushes its real shape (and dtype) into the model's Input node.
3. Set **Validation Split** to `0.2` — batching and the held-out split are data
   concerns, so they live on the node.

In [ ]:
import torch
from torchvision import datasets

# Real MNIST: flatten 28x28 -> 784 and normalize to [0, 1]. Subsample for a
# snappy CPU demo (the full 60k works too, just slower).
mnist = datasets.MNIST(root="./data", train=True, download=True)
X = (mnist.data.float() / 255.0).view(-1, 784)
y = mnist.targets
torch.manual_seed(0)
idx = torch.randperm(len(X))[:12000]
X, y = X[idx], y[idx]

sess.data(X=X, y=y)  # register references with the session (no copies)

## 4. Train from the app

Switch to the **Training** tab:

1. Set the loop config — loss, optimizer, learning rate, epochs, device. (The
   validation split you set on the data node shows up as val metrics here. Set a
   **Seed** to pin runs; left unset, a random one is drawn *and recorded*, so
   every run stays reproducible.)
2. Hit **▶ Run**. Training executes in this kernel — exactly the `train()`
   shown in the preview — and epoch metrics stream in below the code. **■ Stop**
   ends a run early (you keep the partially-trained model).

Then pull the results back here:

In [ ]:
# The run's per-epoch metrics — a dict of lists, ready to plot.
sess.run_status()["state"], sess.history

In [ ]:
# The trained model itself, evaluated on the held-out MNIST test set.
# (.to("cpu") because train() leaves the model on its training device.)
test = datasets.MNIST(root="./data", train=False, download=True)
Xt = (test.data.float() / 255.0).view(-1, 784)
yt = test.targets

model = sess.model.to("cpu").eval()
with torch.no_grad():
    acc = (model(Xt).argmax(dim=-1) == yt).float().mean().item()
print(f"test accuracy: {acc:.3f}")

In [ ]:
# Reproducibility + export. The snapshot records the run's seed, resolved
# device, configs, and the exact generated sources that ran; a checkpoint
# bundles all of that with the trained weights in one self-contained file.
print("seed:", sess.snapshot["seed"], "| device:", sess.snapshot["device"])

sess.save_checkpoint("mnist.pt")
model2, snap = lamplighter.load_checkpoint("mnist.pt")  # rebuilds anywhere — no session needed
with torch.no_grad():
    acc2 = (model2(Xt).argmax(dim=-1) == yt).float().mean().item()
print(f"reloaded-model test accuracy: {acc2:.3f}")  # identical weights, identical accuracy

## 5. Checkpoints — keep the run, come back, train further

The Training tab's **Checkpoints strip** and the calls below drive the same
in-kernel store: name the last run to keep it, **Restore** it later as the
current run, or **▶ Resume** to continue it toward its planned epoch target —
an interrupted run finishes its plan; a finished one takes a new, higher
target. Setting **Autosave Every (epochs)** in the Training form rolls a
resumable `autosave` entry as a long run progresses, so stopping one never
costs the epochs already trained (resume finishes the rest).

The loss chart also rings the epoch with the **lowest validation loss**
(`◦ best @k`) — those weights are captured as they happen, and often beat the
final (possibly overfit) model:

In [ ]:
# The best-epoch model: the weights from the lowest-val-loss epoch, captured
# as the run streamed (None if the run had no validation split). Also loadable
# from a checkpoint file: lamplighter.load_checkpoint("mnist.pt", best=True).
best = sess.best_model
if best is not None:
    with torch.no_grad():
        acc_best = (best(Xt).argmax(dim=-1) == yt).float().mean().item()
    print(f"best-epoch test accuracy: {acc_best:.3f}")

sess.checkpoint("mnist-run")  # keep the last run in the session's store
sess.checkpoints()            # what the app's Checkpoints strip lists

In [ ]:
# Resume continues toward the checkpoint's planned epoch target: for an
# interrupted or autosaved run, sess.resume("name") alone finishes the plan.
# This run completed its plan, so raise the target — 5 more epochs on top of
# what's trained (warm start: fresh optimizer, new recorded seed). Watch the
# Training tab: the curve continues where it left off, numbering included.
meta = next(m for m in sess.checkpoints() if m["name"] == "mnist-run")
sess.resume("mnist-run", epochs=meta["epoch"] + 5)

## 6. Prefer to own the loop in the notebook?

Everything still works the classic way: pull the generated pieces here and call
them yourself. Data always flows through `make_dataloaders()` (built from the
dataset node), so the notebook loop is exactly what the Run button executes.
`train()` returns the same history dict, and its `on_epoch` hook gives you
per-epoch callbacks / early stopping. Inspect the exact sources with
`lamplighter.model_code()` / `lamplighter.data_code()` /
`lamplighter.training_code()`, or **Export model.py** from the titlebar.

In [ ]:
model = lamplighter.build_model()                # fresh GeneratedModel from the canvas
make_dataloaders = lamplighter.build_dataloaders()  # the dataset node's pipeline
train = lamplighter.build_trainer()              # the Training tab's train()

train_loader, val_loader = make_dataloaders(X, y)
history = train(model, train_loader, val_loader=val_loader)  # what ▶ Run executes
len(history["train_loss"])

## 7. Recover a closed tab

Close the browser tab, then run the cell below. The canvas rehydrates from the
backend cache — your design comes back intact.

Even a **kernel restart** doesn't lose the canvas: the design autosaves to
`.lamplighter/graph.json` on every edit, and `lamplighter.start()` restores it
when the backend comes up empty. (`start(persist=False)` for scratch sessions.)

In [ ]:
lamplighter.open_editor()

## 8. Tear down

Stops the server thread. (A kernel restart also stops it.)

In [ ]:
lamplighter.stop()